# Check preprocessing

Sample random subjects, compare six preprocessing variants, and inspect image/mask alignment.
Run with the **rnet** Python environment. Run all cells, then change `SUBJECT_INDEX` and rerun
from that cell to inspect another sampled subject. Edit the view settings and rerun their cell to redraw figures without preprocessing again.
Figures use Plotly's native VS Code renderer (`nbformat>=4.2` required in the kernel).
Plotly provides zoom and hover; Jupyter widgets and embedded HTML are not required.

This is a visual smoke check, not a sensitivity analysis. It uses the normalized dataset's
existing `N4` images; no N4 correction is repeated, no model is trained, and no dataset files
are changed. Intensities use **linear interpolation**; masks use **nearest neighbor**.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import Any

import nibabel as nib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import qualitative
from plotly.subplots import make_subplots
from IPython.display import display

# Works when Jupyter starts in the repository root or notebook folder.
REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/preprocessing/preprocessor.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.loader import RodentDatasets, Rodent
from src.preprocessing.preprocessor import (
    NiftiImage, PreprocessingStep, Preprocessor, Normalize, Resample, Reorient, PadderCutter,
    RandomAffine, GaussianBlur, Noise,
)


## 1. Settings and random subjects

Sampling is reproducible and balanced across the selected datasets: choose normalized subject
IDs first, then one random timepoint per chosen subject. Original IDs are shown for traceability;
repeated original ID strings are not merged. D4 is not included by default.

Every input image, ROI mask and brain mask is first reoriented with `Reorient(ref_img)`,
using the same `example/RARE/TBI_fm_19_49/Anat/TBI_fm_19_49_N4.nii.gz` reference as the other
notebooks. This permutes/flips voxel axes while preserving physical coordinates and voxel values;
it does not register anatomy, resample onto the reference grid, or remove header shear.
The inventory below reports original file geometry; comparison tables use reoriented geometry.

`TARGET_SPACING=None` demonstrates resampling at 1.5 times each scan's native voxel spacing,
keeping the volume manageable. Set an explicit spacing in millimeters to inspect another grid.
`PATCH_SHAPE` is in voxels on the resampled grid. These are inspection settings, not a proposed
training configuration. No automatic tenfold spacing correction or label remapping is applied.

In [ ]:
DATA_ROOT: Path = Path(r"D:\Other files\TESI_CNN\MANUSCRIPT_DATASET\normalized")
REFERENCE_PATH: Path = REPO_ROOT / "example/RARE/TBI_fm_19_49/Anat/TBI_fm_19_49_N4.nii.gz"
ref_img: NiftiImage = nib.load(REFERENCE_PATH)
print(f"Reference orientation: {nib.aff2axcodes(ref_img.affine)} | {REFERENCE_PATH.name}")

DATASETS: tuple[str, ...] = ("D1", "D2", "D3")
SUBJECTS_PER_DATASET: int = 1
SEED: int = 42
TARGET_SPACING: tuple[float, float, float] | None = None
RESAMPLE_FACTOR: float = 1.5
PATCH_SHAPE: tuple[int, int, int] = (96, 96, 96)
TRY_RANDOM_AFFINE: bool = True

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Set DATA_ROOT to your normalized dataset: {DATA_ROOT}")
if SUBJECTS_PER_DATASET < 1:
    raise ValueError("SUBJECTS_PER_DATASET must be positive")

collection: RodentDatasets = RodentDatasets()
for dataset in DATASETS:
    collection.add_normalized_dataset(DATA_ROOT, dataset, image_name="N4")

rng: np.random.Generator = np.random.default_rng(SEED)
selected: list[Rodent] = []
for dataset in DATASETS:
    groups: dict[tuple[str, str], list[Rodent]] = {}
    for key in collection.get_subjects_list():
        scan: Rodent = collection.get_subject(key)
        if scan.subject_group[0] == dataset:
            groups.setdefault(scan.subject_group, []).append(scan)
    subject_keys: list[tuple[str, str]] = sorted(groups)
    count: int = min(SUBJECTS_PER_DATASET, len(subject_keys))
    for index in rng.choice(len(subject_keys), size=count, replace=False):
        timepoints: list[Rodent] = groups[subject_keys[int(index)]]
        selected.append(timepoints[int(rng.integers(len(timepoints)))])

if not selected:
    raise ValueError("No subjects found for DATASETS")
manifest: pd.DataFrame = pd.DataFrame([
    {
        "dataset": scan.subject_group[0], "subject_id": scan.subject_id,
        "original_id": scan.original_id, "timepoint": scan.timepoint,
        "modality": scan.modality, "subgroup": scan.metadata.get("subgroup"),
        "label_configuration": scan.metadata.get("label_configuration"),
        "shape": scan.image.shape,
        "spacing_mm": tuple(np.round(nib.affines.voxel_sizes(scan.image.affine), 4)),
    }
    for scan in selected
])
manifest.index.name = "SUBJECT_INDEX"
display(manifest)

## 2. Choose a subject and run the variants

Only the active subject's processed volumes are kept, so browsing several subjects does not
accumulate all variations in memory. Change the index below and rerun this and subsequent cells.
All variants start from the same reoriented image and both reoriented masks.
Reorientation runs before spacing selection, normalization, resampling, cropping or augmentation.

| Variant | Steps |
|---|---|
| Reoriented N4 | Input after reference-based axis reorientation only |
| Z-score | Whole-volume mean 0, standard deviation 1 |
| Min-max | Whole-volume intensity range 0 to 1 |
| Resampled | Linear intensity resampling on the chosen spacing |
| Prepared | Resample, z-score, center crop/pad to `PATCH_SHAPE` |
| Augmented | Prepared steps, affine rotation/scale when supported, blur, noise |

Augmentations are forced on for inspection and seeded. The current ITK conversion rejects
nonorthogonal grids, including small header deviations in some mouse scans. The notebook
reports and skips affine augmentation for these scans; blur and noise still run. It does not
repair the source geometry. Set `TRY_RANDOM_AFFINE=False` to omit rotation/scale explicitly. Rotation is in degrees, blur in voxels,
and noise in normalized intensity units. Cropping/resampling can remove small labeled regions;
the table reports missing IDs rather than assuming every class must survive. Padding occurs
after normalization, so the prepared volume's mean and standard deviation can change.

In [ ]:
SUBJECT_INDEX: int = 0
active: Rodent = selected[SUBJECT_INDEX]
np.random.seed(SEED + SUBJECT_INDEX)

# Release prior source caches and processed volumes when rerunning this cell.
results: dict[str, tuple[NiftiImage, list[NiftiImage]]] = {}
for scan in selected:
    for image in scan.get_images():
        if image is not None:
            image.uncache()

assert active.brain_mask is not None
input_preprocessor: Preprocessor = Preprocessor([Reorient(ref_img)])
input_image: NiftiImage
input_masks: list[NiftiImage]
input_image, input_masks = input_preprocessor.preprocess(
    active.image, [active.labels, active.brain_mask],
)
assert all(nib.aff2axcodes(image.affine) == nib.aff2axcodes(ref_img.affine)
           for image in [input_image, *input_masks])

spacing: np.ndarray = (
    np.asarray(TARGET_SPACING, dtype=float) if TARGET_SPACING is not None
    else nib.affines.voxel_sizes(input_image.affine) * RESAMPLE_FACTOR
)
resample: Resample = Resample(spacing, interpolation=1)
estimated_shape: np.ndarray = np.ceil(
    np.asarray(input_image.shape) * nib.affines.voxel_sizes(input_image.affine) / spacing
)
if np.prod(estimated_shape) > 20_000_000:
    raise ValueError("Preview would exceed 20 million voxels; use a coarser TARGET_SPACING")

prepared_steps: list[PreprocessingStep] = [resample, Normalize("zscore"), PadderCutter(PATCH_SHAPE)]
directions: np.ndarray = input_image.affine[:3, :3] / nib.affines.voxel_sizes(input_image.affine)
orthogonal: bool = bool(np.allclose(directions.T @ directions, np.eye(3), atol=1e-5))
affine_steps: list[PreprocessingStep] = []
if TRY_RANDOM_AFFINE and orthogonal:
    affine_steps.append(RandomAffine(scale_range=(0.95, 1.05), rotation_range=(-8, 8), probability=1))
affine_status: str = "applied" if affine_steps else (
    "skipped: nonorthogonal header directions" if TRY_RANDOM_AFFINE else "disabled in settings"
)
print(f"Affine augmentation: {affine_status}")
pipelines: dict[str, Preprocessor] = {
    "Reoriented N4": Preprocessor([]),
    "Z-score": Preprocessor([Normalize("zscore")]),
    "Min-max": Preprocessor([Normalize("minmax")]),
    "Resampled": Preprocessor([resample]),
    "Prepared": Preprocessor(prepared_steps),
    "Augmented": Preprocessor([
        *prepared_steps,
        *affine_steps,
        GaussianBlur((0.3, 0.6), probability=1),
        Noise((0.03, 0.06), probability=1),
    ]),
}
for name, pipeline in pipelines.items():
    results[name] = pipeline.preprocess(input_image, input_masks)

print(f"{active.subject_group[0]} / {active.id} | original ID: {active.original_id}")
print(f"Orientation: {nib.aff2axcodes(active.image.affine)} -> {nib.aff2axcodes(input_image.affine)}")
print(f"Resampled spacing (mm): {spacing.round(4)}")

## 3. Numeric checks

These assertions catch nonfinite intensities, fractional/new mask IDs, and mismatched image/mask
grids. They do not establish anatomical correctness; inspect the overlays as well. Statistics
include background. ROI coverage is the percentage of nonzero voxels in the output grid.

In [ ]:
def inspect_result(
    name: str, image: NiftiImage, masks: list[NiftiImage],
    original_masks: list[NiftiImage],
) -> dict[str, Any]:
    """Validate geometry and discrete IDs, returning one readable summary row."""
    values: np.ndarray = image.get_fdata()
    assert np.isfinite(values).all(), f"{name}: nonfinite intensities"
    for mask, original in zip(masks, original_masks):
        labels: np.ndarray = mask.get_fdata()
        assert mask.shape == image.shape, f"{name}: mismatched mask shape"
        assert np.allclose(mask.affine, image.affine, rtol=0, atol=1e-6), f"{name}: mismatched affine"
        assert np.isfinite(labels).all() and np.equal(labels, np.rint(labels)).all(), f"{name}: invalid mask IDs"
        allowed: np.ndarray = np.union1d(np.unique(original.get_fdata()), [0])
        assert np.isin(np.unique(labels), allowed).all(), f"{name}: new mask IDs"
    roi: np.ndarray = masks[0].get_fdata()
    ids: np.ndarray = np.unique(roi).astype(int)
    missing: np.ndarray = np.setdiff1d(np.unique(original_masks[0].get_fdata()), ids)
    return {
        "variant": name, "shape": image.shape,
        "spacing_mm": tuple(np.round(nib.affines.voxel_sizes(image.affine), 4)),
        "min": float(values.min()), "max": float(values.max()),
        "mean": float(values.mean()), "std": float(values.std()),
        "ROI_ids": ids.tolist(), "missing_ROI_ids": missing.astype(int).tolist(),
        "ROI_percent": float(100 * np.count_nonzero(roi) / roi.size),
        "checks": "passed",
    }

summary: pd.DataFrame = pd.DataFrame([
    inspect_result(name, image, masks, input_masks)
    for name, (image, masks) in results.items()
]).set_index("variant")
display(summary.round(4))

## 4. Slice comparison

Set `VIEW_AXIS`, `VIEW_POSITION` and `VIEW_OVERLAY` in the view cell below, then rerun that cell.
`VIEW_POSITION=None` selects the slice with the most annotated voxels along the chosen axis;
use a number from 0 to 1 for a manual position through the input. Each panel shows the nearest
slice to the same physical point; an out-of-field point is marked instead of silently clamped.
In-plane axes use millimeters measured from the reoriented input origin, with equal physical aspect ratio
and shared limits covering every output, including padded regions.
Voxel axes now follow the reference orientation. These remain acquisition planes after axis
permutation/flipping, not a registration or an oblique reslice to the reference image.

ROI colors represent the **original mask IDs**, consistently across panels; hover to read them.
The cyan contour shows the brain mask boundary. Intensities use a separate 1st-99th percentile
window per panel: z-score/min-max should look similar to the input despite different numbers.
Use the table above to check normalization. The initial axis-2 slice has the most annotated voxels.
The augmented panel intentionally moves anatomy within its fixed grid.

In [ ]:
def comparison_figure(axis: int = 2, position: float = 0.5, overlay: str = "ROI + brain") -> go.Figure:
    """Compare reoriented slices at a shared physical point with stable categorical colors."""
    if axis not in (0, 1, 2):
        raise ValueError("axis must be 0, 1, or 2")
    if not np.isfinite(position) or not 0 <= position <= 1:
        raise ValueError("position must be between 0 and 1")
    if overlay not in ("ROI + brain", "ROI", "brain", "none"):
        raise ValueError("overlay must be ROI + brain, ROI, brain, or none")
    source: NiftiImage = input_image
    point: np.ndarray = (np.asarray(source.shape) - 1) / 2
    point[axis] = position * (source.shape[axis] - 1)
    world: np.ndarray = nib.affines.apply_affine(source.affine, point)
    horizontal, vertical = [dimension for dimension in range(3) if dimension != axis]
    source_spacing: np.ndarray = nib.affines.voxel_sizes(source.affine)
    directions: np.ndarray = source.affine[:3, :3] / source_spacing
    ids: np.ndarray = np.unique(input_masks[0].get_fdata()).astype(int)
    ids = ids[ids != 0]
    colors: list[str] = qualitative.Alphabet
    count: int = max(len(ids), 1)
    colorscale: list[tuple[float, str]] = [
        (edge / count, colors[index % len(colors)])
        for index in range(count) for edge in (index, index + 1)
    ]
    bounds: dict[int, tuple[float, float]] = {}
    for dimension in (horizontal, vertical):
        edges: list[float] = []
        for image, _ in results.values():
            offset = np.linalg.solve(directions, image.affine[:3, 3] - source.affine[:3, 3])
            step: float = float(nib.affines.voxel_sizes(image.affine)[dimension])
            edges.extend([offset[dimension] - step / 2,
                          offset[dimension] + (image.shape[dimension] - 0.5) * step])
        bounds[dimension] = (min(edges), max(edges))
    figure: go.Figure = make_subplots(rows=2, cols=3, subplot_titles=list(results))
    for panel, (name, (image, masks)) in enumerate(results.items()):
        row, column = panel // 3 + 1, panel % 3 + 1
        voxel: np.ndarray = nib.affines.apply_affine(np.linalg.inv(image.affine), world)
        index: int = int(np.rint(voxel[axis]))
        if not 0 <= index < image.shape[axis]:
            figure.layout.annotations[panel].text += " (outside field of view)"
            continue
        spacing: np.ndarray = nib.affines.voxel_sizes(image.affine)
        offset: np.ndarray = np.linalg.solve(directions, image.affine[:3, 3] - source.affine[:3, 3])
        x: np.ndarray = offset[horizontal] + np.arange(image.shape[horizontal]) * spacing[horizontal]
        y: np.ndarray = offset[vertical] + np.arange(image.shape[vertical]) * spacing[vertical]
        pixels: np.ndarray = np.take(image.get_fdata(), index, axis=axis).T
        low, high = np.percentile(pixels, [1, 99])
        figure.add_trace(go.Heatmap(
            x=x, y=y, z=pixels, colorscale="Gray", zmin=low, zmax=max(high, low + 1e-8),
            showscale=False, hovertemplate="intensity=%{z:.3f}<extra></extra>",
        ), row=row, col=column)
        if "ROI" in overlay and len(ids):
            roi: np.ndarray = np.take(masks[0].get_fdata(), index, axis=axis).T
            categories: np.ndarray = np.where(roi != 0, np.searchsorted(ids, roi), np.nan)
            figure.add_trace(go.Heatmap(
                x=x, y=y, z=categories, customdata=roi, zmin=-0.5, zmax=len(ids) - 0.5,
                colorscale=colorscale, opacity=0.55, hoverongaps=False,
                showscale=name == "Reoriented N4",
                colorbar=dict(title="ROI ID", tickvals=list(range(len(ids))), ticktext=ids.tolist()),
                hovertemplate="ROI ID=%{customdata:.0f}<extra></extra>",
            ), row=row, col=column)
        if "brain" in overlay:
            brain: np.ndarray = np.take(masks[1].get_fdata(), index, axis=axis).T
            figure.add_trace(go.Contour(
                x=x, y=y, z=(brain != 0).astype(float),
                contours=dict(start=0.5, end=0.5, size=1, coloring="none"),
                line=dict(color="cyan", width=1), showscale=False, hoverinfo="skip",
            ), row=row, col=column)
        xaxis: str = "x" if panel == 0 else f"x{panel + 1}"
        figure.update_xaxes(title_text=f"Voxel axis {horizontal} (mm)",
                            range=bounds[horizontal],
                            row=row, col=column)
        figure.update_yaxes(title_text=f"Voxel axis {vertical} (mm)", scaleanchor=xaxis, scaleratio=1,
                            range=bounds[vertical],
                            row=row, col=column)
    figure.update_layout(height=760, title=f"{active.id} | voxel axis {axis} | input position {position:.0%}",
                         template="plotly_white", margin=dict(l=40, r=70, t=90, b=40))
    return figure


def show_comparison(axis: int, position: float, overlay: str) -> None:
    """Display a native Plotly figure through VS Code's notebook renderer."""
    comparison_figure(axis, position, overlay).show(renderer="vscode")

In [ ]:
# Edit these settings, then rerun only this cell to inspect another view.
VIEW_AXIS: int = 2                       # Reference-oriented voxel axis: 0, 1, or 2.
VIEW_POSITION: float | None = None        # None: most annotated slice; otherwise 0 to 1.
VIEW_OVERLAY: str = "ROI + brain"         # "ROI + brain", "ROI", "brain", or "none".

if VIEW_AXIS not in (0, 1, 2):
    raise ValueError("VIEW_AXIS must be 0, 1, or 2")
roi_counts: np.ndarray = np.count_nonzero(
    input_masks[0].get_fdata(), axis=tuple(axis for axis in range(3) if axis != VIEW_AXIS),
)
actual_position: float = (
    float(np.argmax(roi_counts) / max(input_image.shape[VIEW_AXIS] - 1, 1))
    if VIEW_POSITION is None and roi_counts.any()
    else 0.5 if VIEW_POSITION is None else float(VIEW_POSITION)
)
show_comparison(VIEW_AXIS, actual_position, VIEW_OVERLAY)


## 5. Optional: save this check

Set `SAVE_REPORT=True` to write the sampled IDs, numeric table, pipeline configuration and an
offline HTML comparison under `outputs/check_preprocessing/<scan ID>/`. The HTML records the
current view settings as one fixed comparison. Rerun the view cell before exporting changed settings.
Review mask boundaries, small structures, lost labels and crop coverage before changing a
training pipeline. A passing numeric check alone is not sufficient.

In [ ]:
SAVE_REPORT: bool = False
if SAVE_REPORT:
    import json

    report_dir: Path = REPO_ROOT / "outputs" / "check_preprocessing" / active.id
    report_dir.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(report_dir / "sampled_subjects.csv", index=True)
    summary.to_csv(report_dir / "checks.csv")
    configuration: dict[str, Any] = {
        "seed": SEED, "augmentation_seed": SEED + SUBJECT_INDEX,
        "data_root": str(DATA_ROOT), "active_scan": active.id,
        "original_id": active.original_id, "image_path": str(active.image_path),
        "reference_path": str(REFERENCE_PATH),
        "input_preprocessing": json.loads(input_preprocessor.get_configuration()),
        "affine_augmentation": affine_status,
        "view": {"axis": VIEW_AXIS, "position": actual_position,
                 "overlay": VIEW_OVERLAY},
        "pipelines": {name: json.loads(pipeline.get_configuration()) for name, pipeline in pipelines.items()},
    }
    (report_dir / "configuration.json").write_text(json.dumps(configuration, indent=2), encoding="utf-8")
    comparison_figure(VIEW_AXIS, actual_position, VIEW_OVERLAY).write_html(
        report_dir / "comparison.html", include_plotlyjs=True,
    )
    print(f"Saved: {report_dir}")